In [ ]:
import random
from pathlib import Path
from datasets import load_dataset
from tqdm.auto import tqdm

OUT_DIR = Path("/kaggle/working/corpus")

SEED = 42
MIN_CHARS = 200
SHUFFLE_BUFFER = 100_000

GB = 1024 ** 3
EOT = "\n<eos>\n"

sources = {
    "fineweb_edu": (5.0, "HuggingFaceFW/fineweb-edu", "sample-10BT"),
    "dclm": (3.0, "mlfoundations/dclm-baseline-1.0-parquet", None),
    "cosmopedia_v2": (3.0, "HuggingFaceTB/smollm-corpus", "cosmopedia-v2"),
    "bookcorpus": (3.0, "rojagtap/bookcorpus", None),
    "wikipedia": (2.0, "wikimedia/wikipedia", "20231101.en"),
}

def stream_dataset(repo, config=None):
    ds = load_dataset(repo, config, split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=SHUFFLE_BUFFER)

    for row in ds:
        text = row.get("text")
        if text:
            yield text

def build_dataset(name, size_gb, repo, config=None):
    output = OUT_DIR / f"{name}.txt"
    if output.exists():
        print(f"{name} already exists")
        return

    target = int(size_gb * GB)
    written = 0
    print(f"\n{name} target: {size_gb:.1f} GB")
    stream = stream_dataset(repo, config)

    with open(output, "wb") as f:
        with tqdm(total=target, unit="B", unit_scale=True, desc=name) as pbar:
            for text in stream:
                text = text.strip()
                if len(text) < MIN_CHARS:
                    continue

                text = text.replace("\x00", " ").replace("<eos>", "")
                if len(text) < MIN_CHARS:
                    continue

                data = (text + EOT).encode("utf-8")
                f.write(data)

                written += len(data)
                pbar.update(len(data))
                if written >= target:
                    break

    if written == 0:
        output.unlink(missing_ok=True)
        raise RuntimeError("No usable text was found.")
    print(f"{name}: {written / GB:.2f} GB")

OUT_DIR.mkdir(parents=True, exist_ok=True)

for name, (size, repo, config) in sources.items():
    try:
        build_dataset(name, size, repo, config)
    except Exception as e:
        print(f"{name} failed: {e}")

print("\nCorpus size:")
total = 0

for name in sources:
    file = OUT_DIR / f"{name}.txt"
    if file.exists():
        size = file.stat().st_size / GB
        total += size
        print(f"{name:<20} {size:.2f} GB")

print(f"{'Total':<20} {total:.2f} GB")


fineweb_edu - target: 5.0 GB


fineweb_edu:   0%|          | 0.00/5.37G [00:00<?, ?B/s]

README.md:   0%|          | 0.00/26.4k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

fineweb_edu: 5.00 GB

dclm - target: 3.0 GB


dclm:   0%|          | 0.00/3.22G [00:00<?, ?B/s]

... (11 outputs left out)

wikipedia:   0%|          | 0.00/2.15G [00:00<?, ?B/s]

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

wikipedia: 2.00 GB

Corpus size:
fineweb_edu          5.00 GB
dclm                 3.00 GB
cosmopedia_v2        3.00 GB
bookcorpus           3.00 GB
wikipedia            2.00 GB
Total                16.00 GB


In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

CORPUS_DIR = Path("/kaggle/working/corpus")
TOKENIZER_PATH = "/kaggle/working/tokenizer.json"

SAMPLE_RATIO = 0.25  # ~4 GB of a 16 GB corpus
VOCAB_SIZE = 34_000
MIN_FREQUENCY = 2
SEED = 42
EOT = "\n<eos>\n"
CHUNK_CHARS = 1 << 24 

SPECIAL_TOKENS = ["<pad>", "<unk>", "<bos>", "<eos>"]

files = sorted(CORPUS_DIR.glob("*.txt"))

def iter_docs(path):
    """Yield documents split on EOT, reading in fixed-size chunks."""
    tail = ""

    with open(path, "r", encoding="utf-8", newline="") as f:
        while True:
            chunk = f.read(CHUNK_CHARS)
            if not chunk:
                break
            tail += chunk
            parts = tail.split(EOT)
            tail = parts.pop()
            
            yield from parts

    if tail.strip():
        yield tail

def text_stream():
    for file in files:
        rng = random.Random(f"{SEED}-{file.name}")
        kept = total = 0  
        print(f"Sampling {SAMPLE_RATIO:.1%} of documents from {file.name}")

        for doc in iter_docs(file):
            total += 1
            if rng.random() < SAMPLE_RATIO:
                kept += 1
                yield doc.replace("\x00", " ").replace("<eos>", "")
        print(f"  kept {kept:,} of {total:,} documents")

tokenizer = Tokenizer(BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
tokenizer.decoder = ByteLevelDecoder()

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    initial_alphabet=ByteLevel.alphabet(),
    special_tokens=SPECIAL_TOKENS,
)

tokenizer.train_from_iterator(text_stream(), trainer=trainer)
tokenizer.save(TOKENIZER_PATH)

print("\nTokenizer saved to:", TOKENIZER_PATH)
print(f"Vocabulary size: {tokenizer.get_vocab_size()}")

Sampling 25.0% of documents from bookcorpus.txt
  kept 64,170 of 256,704 documents
Sampling 25.0% of documents from cosmopedia_v2.txt
  kept 214,603 of 856,464 documents
Sampling 25.0% of documents from dclm.txt
  kept 140,581 of 563,621 documents
Sampling 25.0% of documents from fineweb_edu.txt
  kept 283,455 of 1,132,535 documents
Sampling 25.0% of documents from wikipedia.txt
  kept 167,860 of 673,826 documents


Tokenizer saved to: /kaggle/working/tokenizer.json
Vocabulary size: 34000


In [4]:
eos_id = tokenizer.token_to_id("<eos>")
ids = tokenizer.encode("Hello world.<eos>").ids

assert eos_id is not None, "<eos> must exist in vocabulary"
assert ids.count(eos_id) == 1, "<eos> should encode to exactly one token"

sample = "Hello, world! café 你好 🙂"
decoded = tokenizer.decode(tokenizer.encode(sample).ids)

print("Special token ids:", {t: tokenizer.token_to_id(t) for t in SPECIAL_TOKENS})
print("Round trip OK:", decoded == sample)

Special token ids: {'<pad>': 0, '<unk>': 1, '<bos>': 2, '<eos>': 3}
Round trip OK: True


In [5]:
import numpy as np

BIN_DIR = Path("/kaggle/working/bins")
TRAIN_DIR = BIN_DIR / "train"
VAL_DIR = BIN_DIR / "val"

TRAIN_DIR.mkdir(parents=True, exist_ok=True)
VAL_DIR.mkdir(parents=True, exist_ok=True)

VAL_RATIO = 0.005
BATCH_DOCS = 512

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

EOS_ID = tokenizer.token_to_id("<eos>")
assert EOS_ID is not None, "<eos> must exist in vocabulary"

files = sorted(CORPUS_DIR.glob("*.txt"), key=lambda p: p.stat().st_size)

def batches(docs, n):
    batch = []
    for doc in docs:
        batch.append(doc)
        if len(batch) == n:
            yield batch
            batch = []
    if batch:
        yield batch

def write_batch(batch, rng, ftrain, fval):
    train, val = [], []
    for enc in tokenizer.encode_batch(batch):
        ids = enc.ids + [EOS_ID]
        (val if rng.random() < VAL_RATIO else train).extend(ids)

    np.array(train, dtype=np.uint16).tofile(ftrain)
    np.array(val, dtype=np.uint16).tofile(fval)
    return len(train), len(val)

def tokenize_file(file):
    rng = random.Random(f"{SEED}-split-{file.name}")
    n_train = n_val = 0

    train_path = TRAIN_DIR / f"{file.stem}.bin"
    val_path = VAL_DIR / f"{file.stem}.bin"

    with open(train_path, "wb") as ftrain, open(val_path, "wb") as fval:
        with tqdm(
            total=file.stat().st_size,
            unit="B",
            unit_scale=True,
            desc=file.stem,
        ) as pbar:

            for batch in batches(iter_docs(file), BATCH_DOCS):
                t, v = write_batch(batch, rng, ftrain, fval)

                n_train += t
                n_val += v

                pbar.update(sum(len(d.encode("utf-8")) + len(EOT) for d in batch))
    return n_train, n_val

total_train = total_val = 0

for file in files:
    n_train, n_val = tokenize_file(file)
    file.unlink()

    total_train += n_train
    total_val += n_val
    print(f"{file.stem:<16} train {n_train / 1e6:8.1f}M  val {n_val / 1e6:6.2f}M tokens")
    
print(f"\n{'Total':<16} train {total_train / 1e9:.2f}B  val {total_val / 1e6:.1f}M tokens")

size = sum(p.stat().st_size for p in BIN_DIR.rglob("*.bin")) / GB
print(f"Bin size: {size:.2f} GB")

wikipedia:   0%|          | 0.00/2.15G [00:00<?, ?B/s]

wikipedia        train    514.5M  val   2.53M tokens


cosmopedia_v2:   0%|          | 0.00/3.22G [00:00<?, ?B/s]

cosmopedia_v2    train    623.9M  val   3.12M tokens


dclm:   0%|          | 0.00/3.22G [00:00<?, ?B/s]

dclm             train    751.7M  val   3.95M tokens


bookcorpus:   0%|          | 0.00/3.22G [00:00<?, ?B/s]

bookcorpus       train    742.4M  val   3.70M tokens


fineweb_edu:   0%|          | 0.00/5.37G [00:00<?, ?B/s]

fineweb_edu      train   1166.4M  val   5.83M tokens

Total            train 3.80B  val 19.1M tokens
Bin size: 7.11 GB
